### 06. Model Inference and Submission Generation

This notebook contains the interface code to:
1. Load the raw test data (`estate_test.csv`) from the root folder.
2. Apply feature engineering (binning `PropertyAge` into categories).
3. Load the pre-trained `preprocessor.joblib` pipeline and transform the raw test data.
4. Load the best-performing model (`xgboost_model.joblib`) to predict housing prices.
5. Format and save the predictions in the standard Kaggle submission format (`PropertyID`, `TargetPrice`).

#### 1. Import Dependencies

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib

#### 2. Load Raw Test Data

In [ ]:
# Load test data from root directory
test_df = pd.read_csv("../estate_test.csv")
print(f"Loaded test dataset shape: {test_df.shape}")
test_df.head()

#### 3. Preprocess Features (Age Binning)

In [ ]:
# Extract property IDs for submission format
property_ids = test_df["PropertyID"]

# Define the PropertyAge binning function
def property_age_binning(age):
    if age <= 15:
        return 'New'
    elif age <= 35:
        return 'Moderate'
    else:
        return 'Old'

# Apply binning
features = test_df.copy()
features['PropertyAge_bins'] = features['PropertyAge'].apply(property_age_binning)

# Select and order columns as expected by the preprocessor
input_cols = [
    'IncomeLevel', 
    'TotalRooms', 
    'TotalBedrooms', 
    'NeighborhoodPop', 
    'AvgOccupancy', 
    'Latitude', 
    'Longitude', 
    'RoomsPerHousehold', 
    'BedroomsRatio', 
    'PropertyAge_bins'
]
features_subset = features[input_cols]
print("Preprocessed features preview:")
features_subset.head()

#### 4. Load Preprocessor and XGBoost Model

In [ ]:
# Load pre-trained preprocessor and model
preprocessor = joblib.load("models/preprocessor.joblib")
best_model = joblib.load("models/xgboost_model.joblib")
print("Loaded preprocessor and model successfully!")

#### 5. Generate and Post-Process Predictions

In [ ]:
# Transform features using scaling and encoding pipeline
features_processed = preprocessor.transform(features_subset)

# Predict prices
predictions = best_model.predict(features_processed)

# Post-process: Cap negative predictions at 0.0
predictions = np.clip(predictions, 0.0, None)
print(f"Generated predictions for {len(predictions)} properties.")

#### 6. Save Submission Files

In [ ]:
# Create submission DataFrame
submission_df = pd.DataFrame({
    "PropertyID": property_ids,
    "TargetPrice": predictions
})

# 1. Save local backup to artifacts folder
os.makedirs("artifacts", exist_ok=True)
submission_df.to_csv("artifacts/estate_test_predictions.csv", index=False)
print("Saved local backup to: artifacts/estate_test_predictions.csv")

# 2. Save final submission file to the root folder
submission_df.to_csv("../submission.csv", index=False)
print("Saved final Kaggle submission file to: ../submission.csv")

# Preview
print("\nPreview of submission:")
submission_df.head()